In [99]:
from pyspark.sql.functions import *

fact_orders = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/fact_orders"
)

dim_restaurants = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/dim_restaurants"
)

In [100]:
restaurant_metrics = (
    fact_orders
    .groupBy("restaurant_id")
    .agg(
        count("order_id").alias("total_orders"),
        sum("sales_amount").alias("total_revenue"),
        avg("sales_amount").alias("avg_order_value"),
        sum("sales_qty").alias("total_quantity")
    )
)

In [101]:
restaurant_intelligence = (
    restaurant_metrics
    .join(
        dim_restaurants,
        "restaurant_id",
        "left"
    )
)

In [102]:
from pyspark.sql.window import Window

revenue_window = Window.orderBy(
    col("total_revenue").desc()
)

restaurant_intelligence = (
    restaurant_intelligence
    .withColumn(
        "revenue_rank",
        dense_rank().over(revenue_window)
    )
)

In [103]:
restaurant_intelligence = (
    restaurant_intelligence
    .withColumn(
        "performance_band",
        when(
            col("total_revenue") >= 50000,
            "Top Performer"
        )
        .when(
            col("total_revenue") >= 10000,
            "Average Performer"
        )
        .otherwise("Underperformer")
    )
)

In [104]:
city_window = Window.partitionBy(
    "city"
).orderBy(
    col("total_revenue").desc()
)

restaurant_intelligence = (
    restaurant_intelligence
    .withColumn(
        "city_rank",
        dense_rank().over(city_window)
    )
)

In [105]:
restaurant_intelligence.printSchema()

In [106]:
display(
    restaurant_intelligence.orderBy(
        col("total_revenue").desc()
    )
)

In [107]:
restaurant_intelligence.groupBy(
    "performance_band"
).count().show()

In [108]:
restaurant_intelligence.write \
    .mode("overwrite") \
    .parquet(
        "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/restaurant_intelligence"
    )

In [109]:
restaurant_check = spark.read.parquet(
    "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/restaurant_intelligence"
)

print("Rows:", restaurant_check.count())

display(
    restaurant_check.orderBy(
        col("total_revenue").desc()
    )
)

In [110]:
print("Rows:", restaurant_check.count())

In [111]:
display(restaurant_check.orderBy(col("total_revenue").desc()))